In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

In [2]:
dataset = pd.read_csv("data/dataset.csv")

In [3]:
sample, reste = train_test_split(
    dataset,
    train_size=50000,
    stratify=dataset["locationid"],
    random_state=42
)

In [4]:
print(sample.shape)
print(reste.shape)

(50000, 8)
(18657457, 8)


In [5]:
print(f"Sample size : {len(sample) / len(dataset) * 100:.2f} %")
print(f"Reste size : {len(reste) / len(dataset) * 100:.2f} %")

Sample size : 0.27 %
Reste size : 99.73 %


In [6]:
sample["date/time"] = pd.to_datetime(sample["date/time"], format="ISO8601")

In [7]:
sample["date/time"].value_counts().sum()

50000

In [8]:
sample.head()

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone
12452807,2015-05-07 10:51:00,40.688514,-73.964933,B02598,Jeudi,49,Brooklyn,Clinton Hill
10841777,2015-06-04 09:58:00,40.727944,-73.985214,B02764,Jeudi,79,Manhattan,East Village
10099438,2015-05-18 12:44:00,40.756816,-73.972145,B02764,Lundi,162,Manhattan,Midtown East
6064725,2015-06-18 01:04:00,40.620962,-73.954604,B02764,Jeudi,165,Brooklyn,Midwood
993497,2014-05-04 08:32:00,40.726000,-73.992900,B02682,Dimanche,114,Manhattan,Greenwich Village South


In [9]:
begin = pd.Timestamp("2014-05-01 00:00:00")
end = begin + pd.Timedelta(days=10)

In [10]:
t1 = sample[(sample["date/time"] >= begin) & (sample["date/time"] < end)].copy()

In [11]:
t1.head()

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone
993497,2014-05-04 08:32:00,40.7260,-73.9929,B02682,Dimanche,114,Manhattan,Greenwich Village South
613077,2014-05-03 18:36:00,40.7713,-73.9896,B02598,Samedi,143,Manhattan,Lincoln Square West
979774,2014-05-02 16:54:00,40.7169,-74.0107,B02682,Vendredi,231,Manhattan,TriBeCa/Civic Center
981402,2014-05-02 19:09:00,40.7322,-73.9925,B02682,Vendredi,113,Manhattan,Greenwich Village North
673575,2014-05-10 22:05:00,40.7041,-74.0089,B02598,Samedi,87,Manhattan,Financial District North


In [18]:
print("Pickups in the selected hour:", len(t1))

fig_one_hour = px.scatter_map(
    t1,
    lat="lat_x",
    lon="lon_x",
    color="borough",
    center={"lat": 40.73, "lon": -73.97},
    hover_name="zone",
    hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge sur 10 jours en fonction des boroughs",
)
fig_one_hour.show()

Pickups in the selected hour: 550


In [13]:
weekday_distrib = dataset.groupby("weekday").agg(
    pickups=("date/time", "size"),
    days=("date/time", "nunique"),
).reset_index().sort_values("weekday")

In [14]:
weekday_distrib = weekday_distrib.sort_values("pickups", ascending=False)
weekday_distrib.head()

,weekday,pickups,days
5,Samedi,3041187,500431
6,Vendredi,3008172,480312
1,Jeudi,2901178,462849
4,Mercredi,2578562,401348
3,Mardi,2525268,378060


In [15]:
fig = px.bar(
    weekday_distrib,
    x="weekday",
    y="pickups",
    text="pickups",
    title="Nombre de prises en charge par jour de la semaine",
    labels={"weekday": "Jour de la semaine", "pickups": "Nombre de prises en charge"},
    color="weekday",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

In [19]:
print("Pickups in the selected hour:", len(t1))

fig_weekday = px.scatter_map(
    t1,
    lat="lat_x",
    lon="lon_x",
    color="weekday",
    center={"lat": 40.73, "lon": -73.97},
    hover_name="zone",
    hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge sur 10 jours en fonction des jours de la semaine",
)
fig_weekday.show()

Pickups in the selected hour: 550
